# Climate Pipeline Agent — Gemini

Type one natural-language prompt and Gemini generates a complete Python pipeline
using **cdh** functions directly: **download → process → visualize**.

**Examples:**
- `"monthly precipitation plot for Ghana 2018"`
- `"annual mean temperature map for Honduras 2019 to 2021"`
- `"accumulated solar radiation for Malawi 2020, spatial map"`
- `"precipitation by district for Kenya 2018, bar chart"`

> **How it works:** Gemini reads the cdh API description in the system prompt
> and generates Python code. That code is executed directly in this notebook.
> No wrapper functions — the generated code calls `fetch_chirps`, `aggregate_temporal`, etc. directly.

In [ ]:
# Install dependencies
!pip install --upgrade setuptools --quiet
!pip install google-generativeai pycountry rasterio rioxarray plotly s3fs zarr --quiet
!pip install git+https://github.com/anaguilarar/CDH_Skills.git --quiet

In [ ]:
import re
import google.generativeai as genai

# Load API key from Colab Secrets
# Add a secret named GOOGLE_API_KEY via the key icon in the left sidebar
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)
print("API key loaded.")

In [ ]:
SYSTEM_PROMPT = """
You are a climate data expert. Given a natural-language request, generate complete executable
Python code that runs the full pipeline: download → process → visualize.

## cdh package — Data Fetching

from cdh import fetch_chirps, fetch_chirts, fetch_nasa_power
from cdh.summarize import aggregate_temporal, monthly_climatology, seasonal_climatology, aggregate_by_admin

### fetch_chirps
fetch_chirps(country, start_date, end_date, output_folder=None) -> xr.Dataset
  - Variable returned: 'pr' (renamed from 'precipitation')
  - Use for: precipitation, rainfall, rain
  - country: full name ("Ghana") or ISO3 code ("GHA")
  - dates: "YYYY-MM-DD"

### fetch_chirts
fetch_chirts(country, start_date, end_date, variables=["tmax","tmin"], output_folder=None) -> xr.Dataset
  - Variables returned: 'tasmax' (renamed from 'tmax'), 'tasmin' (renamed from 'tmin')
  - Use for: temperature, max temperature, min temperature

### fetch_nasa_power
fetch_nasa_power(country, start_date, end_date, variables=["ALLSKY_SFC_SW_DWN","RH2M","WS2M"], output_folder=None) -> xr.Dataset
  - Use for: solar radiation, humidity, wind speed
  - IMPORTANT: variable names in the returned dataset are renamed to CF standard names:
    ALLSKY_SFC_SW_DWN -> 'rsds'   (solar radiation, MJ/m2/day)
    RH2M             -> 'hurs'   (relative humidity, %)
    WS2M             -> 'sfcWind' (wind speed, m/s)
    T2M_MAX          -> 'tasmax'
    T2M_MIN          -> 'tasmin'
    T2M              -> 'tas'
    PRECTOTCORR      -> 'pr'
  - Always use the CF name (e.g. 'rsds') when accessing ds variables after fetch_nasa_power

### Variable routing (apply automatically)
  precipitation / rainfall        -> fetch_chirps    -> variable: 'pr'
  temperature / tmax / tmin       -> fetch_chirts    -> variables: 'tasmax', 'tasmin'
  solar radiation / sunshine      -> fetch_nasa_power(variables=["ALLSKY_SFC_SW_DWN"]) -> variable: 'rsds'
  wind speed                      -> fetch_nasa_power(variables=["WS2M"])              -> variable: 'sfcWind'
  humidity                        -> fetch_nasa_power(variables=["RH2M"])              -> variable: 'hurs'

### Date rules
  year only (e.g. 2018)       -> "2018-01-01" to "2018-12-31"
  year range (e.g. 2015-2020) -> "2015-01-01" to "2020-12-31"

## cdh package — Processing

### aggregate_temporal
aggregate_temporal(ds, freq="monthly", method="mean") -> xr.Dataset
  freq: 'monthly', 'seasonal', 'annual'
  method: 'mean', 'sum', 'max', 'min'
  Use BEFORE time-series plots. Use method='sum' for precipitation totals.

### monthly_climatology
monthly_climatology(ds) -> xr.Dataset
  Returns 12 values, one per calendar month. Output dim: 'month' (values 1-12).
  Call DIRECTLY on the daily dataset — do NOT call aggregate_temporal first.
  Use for: monthly bar charts, annual cycle plots.

### seasonal_climatology
seasonal_climatology(ds) -> xr.Dataset
  Returns DJF/MAM/JJA/SON. Output dim: 'season'.

### aggregate_by_admin
aggregate_by_admin(ds, country_code, adm_level=1, stat="mean") -> pd.DataFrame
  country_code: ISO3 code (e.g. "GHA").
  adm_level: 1=regions/provinces, 2=districts.
  Returns DataFrame with columns: admin_name, time, <cf_variable_name>.
  Use for: regional comparison bar charts.

## Visualization — use Plotly

### Spatial map
import plotly.express as px
da = ds['rsds'].mean(dim='time')   # use CF variable name, e.g. rsds / pr / tasmax
fig = px.imshow(
    da.values,
    x=da.lon.values, y=da.lat.values,
    color_continuous_scale='YlOrRd',  # pr->Blues, tasmax/tasmin->RdYlBu_r, rsds/sfcWind->YlOrRd
    origin='lower', aspect='auto',
    title='...',
    labels={'color': '<units>'}
)
fig.show()

### Time series (after aggregate_temporal)
import plotly.express as px
ts = ds['rsds'].mean(dim=['lat','lon']).to_series()
fig = px.line(x=ts.index, y=ts.values, title='...', labels={'x': 'Date', 'y': '<units>'})
fig.show()

### Monthly bar chart (after monthly_climatology)
import plotly.express as px
clim = monthly_climatology(ds)
vals = clim['rsds'].mean(dim=['lat','lon']).values  # use CF variable name
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig = px.bar(x=months, y=vals, title='...', labels={'x': 'Month', 'y': '<units>'})
fig.show()

### Admin-unit comparison bar chart (after aggregate_by_admin)
import plotly.express as px
summary = df.groupby('admin_name')['rsds'].mean().reset_index().sort_values('rsds', ascending=False)
fig = px.bar(summary, x='admin_name', y='rsds', title='...')
fig.show()

## Output rules
1. Return ONLY executable Python code inside a ```python ... ``` block.
2. Include ALL imports at the top of the code.
3. Use output_folder='/tmp/cdh_data' for all downloads.
4. Store the xr.Dataset as 'ds' and any DataFrame as 'df'.
5. Print ds after downloading so the user can see the dataset summary.
6. ALWAYS use the CF variable name when accessing ds (e.g. ds['rsds'], not ds['ALLSKY_SFC_SW_DWN']).
7. To get the actual variable name dynamically: var = list(ds.data_vars)[0]
"""

model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    system_instruction=SYSTEM_PROMPT,
)
print("Gemini model ready.")

In [ ]:
def extract_python_code(text: str) -> str:
    """Extract code from a ```python ... ``` block."""
    match = re.search(r'```python\n(.*?)```', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()


def run_pipeline(prompt: str) -> str:
    """Generate and execute a full climate pipeline from a natural-language prompt.

    Gemini reads the cdh API description and generates Python code.
    The code is executed directly in this notebook via exec().
    """
    print(f"Prompt: {prompt}")
    print("Generating pipeline code...\n")

    response = model.generate_content(prompt)
    code = extract_python_code(response.text)

    print("=" * 65)
    print("GENERATED CODE (runs cdh functions directly)")
    print("=" * 65)
    print(code)
    print("=" * 65)
    print("\nExecuting...\n")

    exec(code, globals())  # executes in notebook global scope so ds, df, plots all persist
    return code


print("run_pipeline() ready.")

In [ ]:
run_pipeline("monthly precipitation plot for Ghana 2018")

In [ ]:
run_pipeline("annual mean temperature map for Honduras 2019 to 2020")

In [ ]:
run_pipeline("accumulated solar radiation for Malawi 2020, spatial map")

In [ ]:
run_pipeline("precipitation by region for Kenya 2018, bar chart comparing admin units")